### We will quickly write a rag code just for revision

##### Step1 setUp environment variable

In [7]:
import os
from dotenv import load_dotenv

load_dotenv("C:/Users/user/Secret.env",override=True)

True

##### Step 2 Load PDF

In [10]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "telecom_guide.pdf"

loader = PyPDFLoader(pdf_path)

pages = loader.load()

print(f"Loaded {len(pages)} pages from {pdf_path}")
print(f"First 500 chars of first page: {pages[0].page_content[:500]}")



Exception ignored in PyObject_HasAttr(); consider using PyObject_HasAttrWithError(), PyObject_GetOptionalAttr() or PyObject_GetAttr():
Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 491, in _call_with_frames_removed
AttributeError: partially initialized module 'pandas' from 'c:\Users\user\Desktop\AgenticAI_codebasics\.venv\Lib\site-packages\pandas\__init__.py' has no attribute '_pandas_parser_CAPI' (most likely due to a circular import)


Loaded 9 pages from telecom_guide.pdf
First 500 chars of first page: Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


##### Step 3 chunking

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(pages)

print(f"Splitted {len(pages)} pages into {len(chunks)} chunks")
print("\n sample chunk\n")
print(chunks[3].page_content)

Splitted 9 pages into 24 chunks

 sample chunk

Telecom Technical Reference Guide  - Internal Use Only
2. Troubleshooting Connectivity Issues
Connectivity problems are the most common category of customer complaints. A structured diagnostic approach
resolves the majority of cases without escalation.
Step 1  - Verify signal strength. Open the device's status bar or dial *3001#12345#* (iOS) or use a network signal
app (Android) to view the raw signal level in dBm. A signal above -85 dBm is good; between -85 and -100 dBm is
marginal; below -100 dBm is poor. If signal is weak, moving closer to a window or to a higher floor often helps.
Step 2  - Rule out a temporary network glitch. Toggle airplane mode on for 10 seconds then off. This forces the
device to re-register on the network, which resolves many transient issues.
Step 3  - Check APN settings. The Access Point Name (APN) tells the device how to connect to the carrier's data


##### step 4 embedding

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma.from_documents(chunks, embedding_model)

print(f"Vector store created, total {vector_store._collection.count()} vectors stored")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6694.77it/s]


Vector store created, total 48 vectors stored


##### Step 5 Retriever

In [23]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is VoLTE ? how does it help in calling ?"

retrieved_docs = retriever.invoke(test_query)

#retrieved_docs[0].page_content

for i,doc in enumerate(retrieved_docs):
    print(f"\n\n Document {i+1} \n\n")
    print(doc.page_content)
    print()



 Document 1 


Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls are transmitted as data packets over the LTE network using the IMS (IP
Multimedia Subsystem) core. Benefits include HD voice quality (wideband audio at 16 kHz versus the 3.4 kHz of
legacy calls), faster call setup times (under 2 seconds versus 6-8 seconds on 3G), and the ability to use data and
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Options > Voice & Data and select LTE. If the option is absent



 Document 2 


Telecom T

##### Step 7 we wire everything as pipeline using LCEL (langchain expression language)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    reasoning_format="parsed"
    )
#llm.invoke("can u hear me")

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)
#print(combined_docs)

system_prompt = """You are a helpful assistant that answers questions based on the provided context.
If the answer is not contained within the context below, say "I don't know" .
Context: {context}
"""

prompt = ChatPromptTemplate.from_messages([ #the schema we defined here we must provide context and question otherwise we will get error
    ("system",system_prompt),
    ("human","{question}") # the way we write {question} & upar likhe h {context} will be populated when these are called
])

chain = ( {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser() ) # when we put | the output of LHS becomes input of RHS

# BELOW ARE THE DETAILED COMMENT TO UNDERSTAND THE ABOVE PIPELINE DEEPLY

# Below The | operator works like a Unix pipe: the output of each step becomes the input of the next
# chain = (
#     {"context": retriever | format_docs, "question": RunnablePassthrough()} # context will be populated by retriever | format_docs and question will be passed through unchanged i.e "question": RunnablePassthrough()
#     | prompt # the above line of code will create a new dictionary having keys "context" and "question" and their values already prefilled above then prompt we defined above will be executed with this new dictionary and if we see in prompt we needed these 2 context and question
#     | llm # llm will be invoked
#     | StrOutputParser() # ache se parse krke output dega
# )

#below comments are for above code.
#When a dict appears at the start of a chain as in our case ({"context": retriever | format_docs, "question": RunnablePassthrough()}), then LangChain automatically turns it into a RunnableParallel. **Every value in it receives the same input (our question string which we will ask during invoking)**
# after this line {"context": retriever | format_docs, "question": RunnablePassthrough()} it will become {"context": "chunk1 text\n\n---\n\nchunk2 text...", "question": "How do I activate roaming?"}

print("RAG Pipeline created")

#### Step 8 ASK question with the pipeline


In [32]:
question = "How does international roaming work and what charges should I expect?"

print(f"Q: {question}\n")
print("A:", chain.invoke(question))

Q: How does international roaming work and what charges should I expect?

A: Based on the provided context, here is how international roaming works and the expected charges:

**How It Works**
*   **Connection:** When you travel outside your home network’s coverage area, your device connects to a partner network in the visited country.
*   **Authentication & Billing:** The visited network authenticates you via SS7 or Diameter protocols. Your home network validates your subscription and authorizes service. All data, voice, and SMS traffic is tunnelled back to the home network for billing.
*   **Latency:** This tunnelling process adds latency compared to using a local network.
*   **Activation:** Roaming must be enabled on your account **before** departure. You can do this via the MyTelecom app (Plan & Services > International Roaming) or by calling 611. Network-level activation takes up to 15 minutes. If enabled after landing, you may experience a brief period of no service while the HLR